In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile

zip_path = "/content/drive/MyDrive/skincheck/ham10000.zip"
extract_dir = "/content/ham10000"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

print("Extracted to:", extract_dir)


Mounted at /content/drive
Extracted to: /content/ham10000


In [ ]:
import os, shutil, random
import pandas as pd

RAW_DIR = "/content/ham10000"
IMG_DIRS = [
    os.path.join(RAW_DIR, "HAM10000_images_part_1"),
    os.path.join(RAW_DIR, "HAM10000_images_part_2"),
]
META_CSV = os.path.join(RAW_DIR, "HAM10000_metadata.csv")

OUT_ROOT = "/content/skin_binary"
random.seed(42)

os.makedirs(OUT_ROOT, exist_ok=True)

# Label mappings
MALIGNANT = {"mel", "bcc", "akiec"}
BENIGN = {"nv", "bkl", "df", "vasc"}

# Load metadata
df = pd.read_csv(META_CSV)

# Map 7 classes → 2 classes
def dx_to_bin(dx):
    if dx in MALIGNANT:
        return "malignant"
    elif dx in BENIGN:
        return "benign"
    return None

df["bin_label"] = df["dx"].apply(dx_to_bin)
df = df[df["bin_label"].notnull()].reset_index(drop=True)
print("Total benign/malignant:", len(df))

# Attach full image paths
def find_image_path(image_id):
    fname = image_id + ".jpg"
    for d in IMG_DIRS:
        path = os.path.join(d, fname)
        if os.path.exists(path):
            return path
    return None

df["img_path"] = df["image_id"].apply(find_image_path)
df = df[df["img_path"].notnull()].reset_index(drop=True)
print("After filtering missing images:", len(df))

# Shuffle
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)

# Train/Val/Test split
n = len(df)
n_train = int(0.7 * n)
n_val = int(0.15 * n)
n_test = n - n_train - n_val

train_df = df.iloc[:n_train]
val_df   = df.iloc[n_train:n_train+n_val]
test_df  = df.iloc[n_train+n_val:]

splits = [("train", train_df), ("val", val_df), ("test", test_df)]

# Create folder structure + copy images
for split_name, split_df in splits:
    for label in ["benign", "malignant"]:
        os.makedirs(os.path.join(OUT_ROOT, split_name, label), exist_ok=True)

    for _, row in split_df.iterrows():
        src = row["img_path"]
        label = row["bin_label"]
        dst = os.path.join(OUT_ROOT, split_name, label, os.path.basename(src))
        shutil.copyfile(src, dst)

    print(f"{split_name}: {len(split_df)} images")

print("Dataset prepared at:", OUT_ROOT)


Total benign/malignant: 10015
After filtering missing images: 10015
train: 7010 images
val: 1502 images
test: 1503 images
Dataset prepared at: /content/skin_binary


In [ ]:
!pip install torch torchvision pandas


In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

DATA_ROOT = "/content/skin_binary"
MODEL_PATH = "/content/skincheck_resnet18.pth"

# Training hyperparameters
BATCH_SIZE = 16
EPOCHS = 5
LR = 1e-4

# Detect GPU
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", device)

# Image transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Load datasets
train_ds = datasets.ImageFolder(os.path.join(DATA_ROOT, "train"), transform=train_transform)
val_ds   = datasets.ImageFolder(os.path.join(DATA_ROOT, "val"),   transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

class_names = train_ds.classes
print("Classes:", class_names)

# Load pre-trained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

def train_one_epoch(epoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    print(f"Epoch {epoch} - Train Loss: {total_loss/total:.4f}, Train Acc: {correct/total:.4f}")


def validate():
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total
    print(f"Validation Accuracy: {acc:.4f}")
    return acc


best_acc = 0.0

print("\n🚀 Starting Training...\n")

for epoch in range(1, EPOCHS + 1):
    train_one_epoch(epoch)
    val_acc = validate()

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names,
        }, MODEL_PATH)
        print(f"Saved BEST model - ValAcc: {best_acc:.4f}\n")


print(f"\n\n Training complete! Best Val Accuracy: {best_acc:.4f}")
print("Model saved at:", MODEL_PATH)


Using device: cpu
Classes: ['benign', 'malignant']
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 109MB/s]



🚀 Starting Training...

Epoch 1 - Train Loss: 0.3474, Train Acc: 0.8312
Validation Accuracy: 0.8662
Saved BEST model - ValAcc: 0.8662

Epoch 2 - Train Loss: 0.2871, Train Acc: 0.8712
Validation Accuracy: 0.8862
Saved BEST model - ValAcc: 0.8862

Epoch 3 - Train Loss: 0.2597, Train Acc: 0.8842
Validation Accuracy: 0.8901
Saved BEST model - ValAcc: 0.8901



KeyboardInterrupt: 

In [ ]:
import os
print(os.path.exists("/content/skincheck_resnet18.pth"))


True


In [ ]:
import shutil

src = "/content/skincheck_resnet18.pth"
dst = "/content/drive/MyDrive/skincheck/skincheck_resnet18.pth"

shutil.copyfile(src, dst)
print("Copied to:", dst)


Copied to: /content/drive/MyDrive/skincheck/skincheck_resnet18.pth
